In [ ]:
import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
# =============================================================================
# STANDARDIZED STYLE SETTINGS
# =============================================================================
FONTSIZE_TITLE = 14
FONTSIZE_LABEL = 12
FONTSIZE_TICK = 10
FONTSIZE_LEGEND = 10
FIG_WIDTH = 12  # Consistent width

# Color scheme: Blue-Grey palette (matching other plots)
COLORS = cm.get_cmap('viridis')(np.linspace(0, 1, 4))

In [ ]:
def topk_balacc(df, k=10, kind="preproc"):
    # kind: "preproc" expects TMAX/SNR_FILTERING/ARTIFACT_REMOVAL/DWT_DOWNSAMPLING
    #       "hyper"   expects learning_rate/epochs/batch_size/dropout_rate/l2_lambda
    keys = (["TMAX","ARTIFACT_REMOVAL","DWT_DOWNSAMPLING", "SNR_FILTERING", "SNR_THRESHOLD"] if kind=="preproc"
            else ["learning_rate","epochs","batch_size","dropout_rate","l2_lambda"])

    scores = {
        "PRIMA LE DA": "balanced_accuracy_prima_le_da",
        "PRIMA RCS DA": "balanced_accuracy_prima_rcs_da",
        "MP20 LE DA": "balanced_accuracy_mp20_le_da",
        "MP20 RCS LA": "balanced_accuracy_mp20_rcs_la",
        "RB20 RCS LA": "balanced_accuracy_rb20_rcs_la",
        "ALL COMBINED": "balanced_accuracy_combined"
    }
    need = keys + list(scores.values())
    df = df.dropna(subset=[c for c in need if c in df.columns])

    # combined (no extra DataFrame copies besides these two columns)
    df["avg_all"] = df[list(scores.values())].mean(1)

    def show(title, sort_col, cols):
        out = (df.nlargest(k, sort_col)[cols]
                 .assign(Rank=lambda x: np.arange(1, len(x)+1))
                 .set_index("Rank"))
        print("\n" + "="*120 + f"\n{title}\n" + "="*120)
        print(out.to_string())

    # Per-model top-k (balanced accuracy)
    for name, col in scores.items():
        show(f"TOP {k} ({name}) by Balanced Accuracy",
             col, [col] + keys)

    show(f"TOP {k} (ALL 4 MODELS) by Avg Balanced Accuracy",
         "avg_all", ["avg_all"] + list(scores.values()) + keys)

    return df  # contains avg columns if you want to reuse


def save_top10(df, save_path):

    cols = [
        "balanced_accuracy_prima_le_da",
        "balanced_accuracy_prima_rcs_da",
        "balanced_accuracy_mp20_le_da",
        "balanced_accuracy_mp20_rcs_la",
        "balanced_accuracy_rb20_rcs_la",
    ]

    df = df.dropna(subset=cols).copy()

    # Average balanced accuracy across all three
    df["avg_balanced_accuracy_all"] = df[cols].mean(axis=1)

    top10 = df.nlargest(10, "avg_balanced_accuracy_all")

    # Create directory if neededs
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    top10.to_csv(save_path, index=False)
    print(f"Saved Top 10 to: {save_path}")

    return top10


## 1. Preprocessing

In [ ]:
# =============================================================================
# 1. PREPROCESSING
# =============================================================================
data = pd.read_csv('outputs/results/hyperparam_results/results_preprocessing_old.csv')

model_cols = {
    "PRIMA LE DA": "balanced_accuracy_prima_le_da",
    "PRIMA RCS DA": "balanced_accuracy_prima_rcs_da",
    "MP20 LE DA": "balanced_accuracy_mp20_le_da",
    "MP20 RCS LA": "balanced_accuracy_mp20_rcs_la",
    "Mixed Test": "balanced_accuracy_mixed_test"
}

effects = [
    ("DWT_DOWNSAMPLING", [True, False], "Downsampling"),
    ("SNR_FILTERING", [True, False], "SNR Filtering"),
    ("ARTIFACT_REMOVAL", [True, False], "Artifact Removal"),
]

# PLOTTING - adjusted for shared legend
fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH, 4), sharey=True)

# Bar parameters with spacing
n_conditions = len(model_cols)
n_main_conditions = n_conditions - 1  # Exclude Mixed Test from main bars
width = 0.2
group_width = n_main_conditions * width
group_spacing = 0.3  # Space between groups

# Store handles and labels for shared legend (collect ALL 5)
all_handles = []
all_labels = []

for ax, (column, order, title) in zip(axes, effects):
    stats = (
        data
        .groupby(column)
        [list(model_cols.values())]
        .agg(["mean", "std"])
        .reindex(order)
    )

    # Create positions with spacing
    n_groups = len(order)
    x = np.arange(n_groups) * (group_width + group_spacing)

    for i, (label, col) in enumerate(model_cols.items()):
        color = COLORS[i] if i < len(COLORS) else "#808080"
        means = stats[(col, "mean")]
        stds  = stats[(col, "std")]
        
        if label == "Mixed Test":
            # Special handling for Mixed Test - spans only the 3 test conditions
            span_width = width * 3  # Span 3 bars
            span_offset = width * 0.5  # Shift right by half a bar width
            
            bars = ax.bar(
                x + span_offset,
                means,
                span_width,
                color=color,
                alpha=0.5,  # More transparent
                edgecolor='black',
                linewidth=1.5,
                hatch='///',  # Add hatching pattern
                label=label,
                zorder=2  # Behind individual bars but in front of grid
            )
        else:
            # Regular bars for individual validation sets
            offset = (i - (n_main_conditions - 1) / 2) * width
            
            bars = ax.bar(
                x + offset,
                means,
                width,
                yerr=stds,
                color=color,
                alpha=0.8,
                capsize=5,
                label=label,
                zorder=3  # Bars in front of grid and mixed test bar
            )
        
        # Collect handles and labels from first subplot only
        if ax == axes[0]:
            all_handles.append(bars)
            all_labels.append(label)

    ax.set_title(title, fontsize=FONTSIZE_TITLE, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(order, fontsize=FONTSIZE_TICK)
    ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
    ax.grid(True, axis="y", alpha=0.3, zorder=0)  # Grid behind bars
    ax.set_ylim(0.4, 1.05)

axes[0].set_ylabel(r"Balanced Accuracy", fontsize=FONTSIZE_LABEL, fontweight='bold')

# Add single legend to the right of all subplots with all 5 models
fig.legend(all_handles, all_labels, 
           loc='center left', 
           bbox_to_anchor=(1.02, 0.8),
           fontsize=FONTSIZE_LEGEND, 
           frameon=False,
           ncol=1)  # Force single column

plt.tight_layout()
plt.subplots_adjust(right=1.0)
plt.savefig("outputs/figures/HyperparameterTuning_Preprocessing.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------
# Usage (preprocessing.csv)
# ---------------------------
pre = pd.read_csv("outputs/results/hyperparam_results/results_preprocessing.csv")
topk_balacc(pre, k=10, kind="preproc")

# Save top 10 based on combined TEST ALL + PRIMA LE DA
# top10_pre = save_top10(
#     pre,
#     "outputs/results/hyperparam_results/top_10_preprocessing_test_prima.csv",
# )

## 2. Learning

In [ ]:
# =============================================================================
# 2. LEARNING
# =============================================================================
data_learn = pd.read_csv("outputs/results/hyperparam_results/results_learning_old.csv")

metrics = {
    "PRIMA LE DA": ("balanced_accuracy_prima_le_da", COLORS[0]),
    "PRIMA RCS DA": ("balanced_accuracy_prima_rcs_da", COLORS[1]),
    "MP20 LE DA": ("balanced_accuracy_mp20_le_da", COLORS[2]),
    "MP20 RCS LA": ("balanced_accuracy_mp20_rcs_la", COLORS[3]),
    "Mixed Test": ("balanced_accuracy_mixed_test", COLORS[4] if len(COLORS) > 4 else "#808080"),
}

params = ["learning_rate", "epochs", "batch_size", "dropout_rate", "l2_lambda"]
params_names = {
    "learning_rate": "Learning Rate",
    "epochs": "Epochs",
    "batch_size": "Batch Size",
    "dropout_rate": "Dropout Rate",
    "l2_lambda": r"$\boldsymbol{\lambda}$ (L2)",
}

# Load + clean
df = data_learn.copy()
need_cols = params + [v[0] for v in metrics.values()]
df[need_cols] = df[need_cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=need_cols)

print(f"Total rows: {len(df)}")

# Precompute group stats
stats = {}
for p in params:
    gp = df.groupby(p, sort=True)
    for _, (ycol, _) in metrics.items():
        stats[(p, ycol)] = gp[ycol].agg(mean="mean", std="std")

# 2x3 figure
fig, axes = plt.subplots(2, 3, figsize=(FIG_WIDTH, 8), sharey=True)
axes = axes.ravel()

# Bar parameters with spacing
n_metrics = len(metrics)
n_main_metrics = n_metrics - 1  # Exclude Mixed Test from main bars
bar_width = 0.2
group_width = n_main_metrics * bar_width
group_spacing = 0.3  # Space between groups

colors = [metrics[k][1] for k in metrics]
labels = list(metrics.keys())
ycols = [metrics[k][0] for k in metrics]

# Store ALL handles for legend
all_handles = []
all_labels = []

for ax_i, ax in enumerate(axes):
    if ax_i >= len(params):
        # Use 6th subplot space for legend
        ax.axis("off")
        if all_handles:  # Check if we have collected handles
            ax.legend(all_handles, all_labels, loc='upper left',
                     fontsize=FONTSIZE_LEGEND, frameon=False, ncol=1)
        continue

    p = params[ax_i]
    xvals = stats[(p, ycols[0])].index.to_numpy()
    
    # Create positions with spacing
    n_groups = len(xvals)
    x_pos = np.arange(n_groups) * (group_width + group_spacing)

    for j, (label, ycol, color) in enumerate(zip(labels, ycols, colors)):
        g = stats[(p, ycol)].reindex(xvals)
        
        if label == "Mixed Test":
            # Special handling for Mixed Test - spans only the 3 test conditions (indices 1, 2, 3)
            # NO ERROR BARS for combined score
            span_width = bar_width * 3  # Span 3 bars
            span_offset = bar_width * 0.5  # Shift right by half a bar width
            
            bars = ax.bar(
                x_pos + span_offset,
                g["mean"].to_numpy(),
                span_width,
                color=color,
                alpha=0.5,  # More transparent
                edgecolor='black',
                linewidth=1.5,
                hatch='///',  # Add hatching pattern
                label=label,
                zorder=2  # Behind individual bars but in front of grid
            )
        else:
            # Regular bars for individual validation sets
            offset = (j - (n_main_metrics - 1) / 2) * bar_width
            
            bars = ax.bar(
                x_pos + offset,
                g["mean"].to_numpy(),
                width=bar_width,
                yerr=g["std"].to_numpy(),
                capsize=5,
                color=color,
                alpha=0.8,
                label=label,
                zorder=3  # Bars in front of grid and mixed test bar
            )
        
        # Capture ALL handles from first subplot
        if ax_i == 0:
            all_handles.append(bars)
            all_labels.append(label)

    ax.set_title(params_names.get(p, p), fontsize=FONTSIZE_TITLE, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(v) for v in xvals], rotation=15 if p != "learning_rate" else 0,
                       fontsize=FONTSIZE_TICK)
    ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
    ax.grid(True, axis="y", alpha=0.3, zorder=0)  # Grid behind bars
    ax.set_ylim(0.4, 1.05)

axes[0].set_ylabel(r"Balanced Accuracy", fontsize=FONTSIZE_LABEL, fontweight='bold')
axes[3].set_ylabel(r"Balanced Accuracy", fontsize=FONTSIZE_LABEL, fontweight='bold')
plt.savefig("outputs/figures/HyperparameterTuning_Learning.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
learn = pd.read_csv("outputs/results/hyperparam_results/results_learning.csv")
topk_balacc(learn, k=10,  kind="hyper")

# top10_learn = save_top10(
#     learn,
#     "outputs/results/hyperparam_results/top_10_learning.csv",
# )

## Comparison to other Models

In [ ]:
# =============================================================================
# 3. MODEL COMPARISON
# =============================================================================
data_comp = pd.read_csv("outputs/results/model_comparison_results/results_test.csv")

# Calculate average across the 3 test cases
data_comp["balanced_accuracy_test_avg"] = data_comp[[
    "balanced_accuracy_prima_rcs",
    "balanced_accuracy_mp20",
    "balanced_accuracy_mp20_la_rcs"
]].mean(axis=1)

metrics_comp = {
    "PRIMA LE DA": ("balanced_accuracy_prima", COLORS[0]),
    "PRIMA RCS DA": ("balanced_accuracy_prima_rcs", COLORS[1]),
    "MP20 LE DA": ("balanced_accuracy_mp20", COLORS[2]),
    "MP20 RCS LA": ("balanced_accuracy_mp20_la_rcs", COLORS[3]),
    "Test Avg": ("balanced_accuracy_test_avg", COLORS[4] if len(COLORS) > 4 else "#808080"),  # Gray as default
}

models = data_comp["model"].values
n_models = len(models)
n_conditions = len(metrics_comp)

# Add spacing between groups
group_width = n_conditions * 0.25  # Total width of one group
group_spacing = 0.3  # Space between groups
x = np.arange(n_models) * (group_width + group_spacing)  # Position groups with spacing

width = 0.25

fig, ax = plt.subplots(figsize=(12, 4))

for i, (label, (column, color)) in enumerate(metrics_comp.items()):
    offset = (i - (n_conditions - 1) / 2) * width
    
    # Special handling for "Test Avg" bar
    if label == "Test Avg":
        # Make it span across the 3 test cases (indices 1, 2, 3)
        # Position it centered over those 3 bars
        span_width = width * 3 
        span_offset = 0  # Center it
        
        ax.bar(
            x + span_offset,
            data_comp[column],
            span_width,
            label=label,
            color=color,
            alpha=0.5,  # More transparent
            edgecolor='black',
            linewidth=1.5,
            hatch='///',  # Add hatching pattern
            zorder=2  # Behind individual bars but in front of grid
        )
    else:
        ax.bar(
            x + offset,
            data_comp[column],
            width,
            label=label,
            color=color,
            alpha=0.8,
            zorder=3  # Bars in front of grid and avg bar
        )

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=FONTSIZE_TICK)
ax.set_ylabel(r"Balanced Accuracy", fontsize=FONTSIZE_LABEL, fontweight='bold')
ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=FONTSIZE_LEGEND, frameon=False, ncol=2)  # 2 columns for legend
ax.grid(axis="y", alpha=0.3, zorder=0)  # Grid behind bars

plt.tight_layout()
plt.savefig("outputs/figures/Model_Comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# import pandas as pd
# #male pandas table only with balanced accuracy 
# comp = pd.read_csv("outputs/results/model_comparison_results/results_test.csv")
# comp = comp[["model", "balanced_accuracy_prima", "balanced_accuracy_mixed", "balanced_accuracy_prima_rcs", "balanced_accuracy_mp20", "balanced_accuracy_mp20_la_rcs"]]
# comp.to_csv("outputs/results/model_comparison_results/balanced_accuracy_comparison.csv", index=False)

# # print
# print("\n" + "="*120 + "\nMODEL COMPARISON - BALANCED ACCURACY\n" + "="*120)
# print(comp.to_string(index=False))


## Effect of TMAX on Classification Performance

In [ ]:
# load outputs/results/tmax_results/tmax_balanced_accuracy.csv
data = pd.read_csv('outputs/results/hyperparam_results/results_tmax.csv')
tmax_values = data['TMAX']
balanced_accuracy_prima_le_da = data['balanced_accuracy_prima_le_da']
balanced_accuracy_prima_rcs_da = data['balanced_accuracy_prima_rcs_da']
balanced_accuracy_mp20_le_da = data['balanced_accuracy_mp20_le_da']
balanced_accuracy_mp20_rcs_la = data['balanced_accuracy_mp20_rcs_la']
balanced_accuracy_rb20_rcs_la = data['balanced_accuracy_rb20_rcs_la']
balanced_accuracy_combined = data['balanced_accuracy_combined']

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Create directory for figures if it doesn't exist
if not os.path.exists('figures'):
    os.makedirs('figures')

# Load data
file_path = 'outputs/results/hyperparam_results/results_tmax.csv'
data = pd.read_csv(file_path)

# 1. Define model columns to calculate the overall average
model_cols = [
    'balanced_accuracy_prima_le_da', 
    'balanced_accuracy_prima_rcs_da',
    'balanced_accuracy_mp20_le_da', 
    'balanced_accuracy_mp20_rcs_la',
    'balanced_accuracy_rb20_rcs_la'
]

# Calculate average across all models and find Top 3
data['balanced_accuracy_avg'] = data[model_cols].mean(axis=1)
n = 1
best_indices = data['balanced_accuracy_avg'].nlargest(n).index

# Extract TMAX values for coloring
tmax_values = data['TMAX']

# Combine data for axes
# x = PRIMA (both) -> Average of le_da and rcs_da
x_data = (data['balanced_accuracy_prima_le_da'] + data['balanced_accuracy_prima_rcs_da']) / 2
# z = MP20 (both) -> Average of le_da and rcs_la
z_data = (data['balanced_accuracy_mp20_le_da'] + data['balanced_accuracy_mp20_rcs_la']) / 2
# y = RB20
y_data = data['balanced_accuracy_rb20_rcs_la']

# Create figure and 3D axis
fig = plt.figure(figsize=(10, 10), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')

# Create 3D scatter plot
scatter = ax.scatter(
    x_data,
    y_data,
    z_data,
    c=tmax_values,
    cmap='viridis',
    s=100,
    alpha=0.7,
    edgecolors='black',
    linewidth=0.5
)

# 2. Add Highlights and Labels for Top 3 TMAX Values
for idx in best_indices:
    x = x_data.loc[idx]
    y = y_data.loc[idx]
    z = z_data.loc[idx]
    tmax_val = data.loc[idx, 'TMAX']
    avg_ba = data.loc[idx, 'balanced_accuracy_avg']

    # Draw red highlight ring
    ax.scatter(
        [x], [y], [z],
        s=250,
        facecolors='none',
        edgecolors='red',
        linewidth=2,
        alpha=1.0,
        zorder=10
    )

    # Label with TMAX value and Average Accuracy
    ax.text(
        x+0.01, y, z-0.035, 
        f'  {tmax_val:.0f}ms\n',
        fontsize=10,
        color='red',
        fontweight='bold',
        ha='left',
        va='bottom',
        zorder=11
    )

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.5)
cbar.set_label('tmax (ms)', labelpad=10, fontsize=12)
labels = [20] + np.arange(100, 450, 100).tolist()
cbar.set_ticks(labels)

# Set labels to reflect the combined data
ax.set_xlabel('PRIMA (LE DA + RCS DA)', fontweight='bold', fontsize=12)
ax.set_ylabel('RB20 RCS LA', fontweight='bold', fontsize=12) # Note: y is RB20 in your mapping
ax.set_zlabel('MP20 (LE DA + RCS LA)', fontweight='bold', fontsize=12)
ax.yaxis.label.set_rotation(45)
ax.zaxis.label.set_rotation(90)

# Set same scale/range for all axes for better comparison
all_values = np.concatenate([x_data.to_numpy(), y_data.to_numpy(), z_data.to_numpy()])
vmin, vmax = all_values.min(), all_values.max()
padding = (vmax - vmin) * 0.05
ax.set_xlim([vmin - padding, vmax + padding])
ax.set_ylim([vmin - padding, vmax + padding])
ax.set_zlim([vmin - padding, vmax + padding])

ax.grid(True, alpha=0.3)
#ax.view_init(elev=30)
# Save the plot
plt.savefig("outputs/figures/Pareto_TMAX_combined_highlighted.png", dpi=300, bbox_inches="tight")
plt.show()

# Verification output
print(f"Top {n} TMAX values by average balanced accuracy:")
for idx in best_indices:
    print(f"TMAX: {data.loc[idx, 'TMAX']:.1f} ms | Avg Balanced Accuracy: {data.loc[idx, 'balanced_accuracy_avg']:.4f}")

In [ ]:
# show n best tmax for combined score
n = 3
print("\n" + "="*120 + f"\nTOP {n} TMAX VALUES BY COMBINED BALANCED ACCURACY\n" + "="*120)
best_indices = data['balanced_accuracy_combined'].nlargest(n).index
for idx in best_indices:
    tmax = data.loc[idx, 'TMAX']
    ba = data.loc[idx, 'balanced_accuracy_combined']
    print(f"TMAX: {tmax} ms with balanced accuracy {ba:.3f}")

# print for each model separately
models = {
    "PRIMA LE DA": "balanced_accuracy_prima_le_da",
    #"PRIMA RCS DA": "balanced_accuracy_prima_rcs_da",
    "MP20 LE DA": "balanced_accuracy_mp20_le_da",
    "MP20 RCS LA": "balanced_accuracy_mp20_rcs_la",
    #"RB20 RCS LA": "balanced_accuracy_rb20_rcs_la",
}   
print("\n" + "="*120 + f"\nTOP {n} TMAX VALUES BY BALANCED ACCURACY PER MODEL\n" + "="*120)
for model_name, col in models.items():
    print(f"\n--- {model_name} ---")
    best_indices = data[col].nlargest(n).index
    for idx in best_indices:
        tmax = data.loc[idx, 'TMAX']
        ba = data.loc[idx, col]
        print(f"TMAX: {tmax} ms with balanced accuracy {ba:.3f}")

# average of all models
data['balanced_accuracy_avg'] = data[[col for col in models.values()]].mean(axis=1)
print("\n" + "="*120 + f"\nTOP {n} TMAX VALUES BY AVERAGE BALANCED ACCURACY ACROSS ALL MODELS\n" + "="*120)
best_indices = data['balanced_accuracy_avg'].nlargest(n).index
for idx in best_indices:
    tmax = data.loc[idx, 'TMAX']
    ba_avg = data.loc[idx, 'balanced_accuracy_avg']
    print(f"TMAX: {tmax} ms with average balanced accuracy {ba_avg:.3f}")

In [ ]:
# Keep only tmax >= 20 and multiples of 40
mask = (tmax_values >= 10) & (tmax_values % 10 == 0)

tmax_plot = tmax_values[mask]
prima_le = balanced_accuracy_prima_le_da[mask]
prima_rcs = balanced_accuracy_prima_rcs_da[mask]
mp20_le = balanced_accuracy_mp20_le_da[mask]
mp20_rcs = balanced_accuracy_mp20_rcs_la[mask]

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 4))

ax.plot(tmax_plot, prima_le, marker='o', label='PRIMA LE DA', color=COLORS[0], linewidth=2.5, markersize=8)
ax.plot(tmax_plot, prima_rcs, marker='o', label='PRIMA RCS DA', color=COLORS[1], linewidth=2.5, markersize=8)
ax.plot(tmax_plot, mp20_le, marker='o', label='MP20 LE DA', color=COLORS[2], linewidth=2.5, markersize=8)
ax.plot(tmax_plot, mp20_rcs, marker='o', label='MP20 RCS LA', color=COLORS[3], linewidth=2.5, markersize=8)
ax.plot(tmax_plot, balanced_accuracy_rb20_rcs_la[mask], marker='o', label='RB20 RCS LA', color=COLORS[4] if len(COLORS) > 4 else "#808080", linewidth=2.5, markersize=8)

ax.set_xlabel(r'Signal Length (ms)', fontsize=FONTSIZE_LABEL, fontweight='bold')
ax.set_ylabel(r'Balanced Accuracy', fontsize=FONTSIZE_LABEL, fontweight='bold')

ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax.legend(fontsize=FONTSIZE_LEGEND, frameon=False, loc='best')
ax.grid(True, alpha=0.3, zorder=0)
ax.set_ylim(0.4, 1.05)

plt.tight_layout()
plt.savefig("outputs/figures/BA_vs_TMAX.png", dpi=300, bbox_inches="tight")
plt.show()